# 03. Modeling (PROFESSIONAL VERSION)

Objetivo:
 - Entrenar modelos predictivos sin data leakage
 - Evaluar baseline vs modelo avanzado
 - Obtener métricas robustas

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

dataPath = Path("../data/processed/model_dataset.csv")

df = pd.read_csv(dataPath, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Dataset cargado:", df.shape)

Dataset cargado: (47769, 24)


## 1. Split temporal (CRÍTICO)

In [6]:
splitDate = "2018-01-01"

trainDf = df[df["date"] < splitDate].copy()
testDf = df[df["date"] >= splitDate].copy()

print("Train:", trainDf.shape)
print("Test:", testDf.shape)

Train: (40016, 24)
Test: (7753, 24)


## 2. Preparación de features

In [7]:
dropCols = ["date", "homeTeam", "awayTeam", "target"]

X_train = trainDf.drop(columns=dropCols)
X_test = testDf.drop(columns=dropCols)

y_train = trainDf["target"]
y_test = testDf["target"]

# Encode target
targetMap = {"homeWin": 0, "draw": 1, "awayWin": 2}
y_train = y_train.map(targetMap)
y_test = y_test.map(targetMap)

print("Features:", X_train.shape)

Features: (40016, 20)


## 3. Baseline — Logistic Regression

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logModel = LogisticRegression(max_iter=1000, multi_class="multinomial")
logModel.fit(X_train_scaled, y_train)

y_pred_log = logModel.predict(X_test_scaled)
y_prob_log = logModel.predict_proba(X_test_scaled)

# %%
print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("Log Loss:", log_loss(y_test, y_prob_log))
print(classification_report(y_test, y_pred_log))

c:\Users\restr\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


=== Logistic Regression ===
Accuracy: 0.5942215916419451
Log Loss: 0.8799107806701256
              precision    recall  f1-score   support

           0       0.61      0.86      0.72      3691
           1       0.30      0.03      0.06      1789
           2       0.58      0.60      0.59      2273

    accuracy                           0.59      7753
   macro avg       0.50      0.50      0.46      7753
weighted avg       0.53      0.59      0.53      7753



## 4. Modelo avanzado — XGBoost

In [10]:
xgbModel = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

xgbModel.fit(X_train, y_train)

y_pred_xgb = xgbModel.predict(X_test)
y_prob_xgb = xgbModel.predict_proba(X_test)

# %%
print("=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Log Loss:", log_loss(y_test, y_prob_xgb))
print(classification_report(y_test, y_pred_xgb))

=== XGBoost ===
Accuracy: 0.5970592028892042
Log Loss: 0.8855609969866844
              precision    recall  f1-score   support

           0       0.62      0.86      0.72      3691
           1       0.35      0.05      0.09      1789
           2       0.57      0.61      0.59      2273

    accuracy                           0.60      7753
   macro avg       0.51      0.50      0.47      7753
weighted avg       0.54      0.60      0.54      7753



## 5. Comparación final

In [12]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "LogLoss": [
        log_loss(y_test, y_prob_log),
        log_loss(y_test, y_prob_xgb)
    ]
})

results

,Model,Accuracy,LogLoss
0,Logistic Regression,0.594222,0.879911
1,XGBoost,0.597059,0.885561
